# 📊 Barcode Recovery Pipeline — исследовательская тетрадь

**Цель:** систематически перебрать комбинации предобработки на кропах штрихкодов
от YOLO и оценить каждую не только по факту декодирования, но и по метрикам качества.

**Структура:**
1. Установка и импорты
2. Загрузка кропов
3. Базовые метрики качества (без декодирования)
4. Выравнивание (deskew)
5. Классические алгоритмы предобработки
6. SR-модели: EDSR, IFAN, DRBNet
7. Комбинированные пайплайны
8. Итоговая таблица результатов
9. Визуализация лучших/худших случаев

## 1. Установка зависимостей

In [ ]:
# Запусти один раз, потом можно закомментировать
import subprocess
import sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip('pyzbar', 'zxingcpp', 'opencv-contrib-python', 'scikit-image',
    'matplotlib', 'pandas', 'tqdm', 'Pillow', 'super-image',
    'torch', 'torchvision', 'einops', 'timm')

print('✓ все пакеты установлены')

## 2. Импорты и конфиг

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from tqdm.notebook import tqdm
from dataclasses import dataclass, field
from typing import Optional, Callable
import warnings
warnings.filterwarnings('ignore')

# Декодеры
from pyzbar import pyzbar
import zxingcpp

# Метрики изображения
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 9

print('✓ импорты OK')

In [ ]:
# ── КОНФИГ — измени под свой путь ──────────────────────────────────────────
CROPS_DIR = Path('/home/morph/projects-wsl/lenta_hack/sol_main/data/testdata/barcode')   # папка с кропами от YOLO
RESULTS_DIR = Path('./results')
RESULTS_DIR.mkdir(exist_ok=True)

# Пути к весам нейросетей (заполни после скачивания)
IFAN_REPO    = Path('./IFAN')          # git clone https://github.com/codeslake/IFAN
DRBNET_REPO  = Path('./DRBNet')        # git clone https://github.com/lingyanruan/DRBNet

# Форматы изображений которые ищем
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

print(f'Папка кропов: {CROPS_DIR.resolve()}')
print(f'Существует: {CROPS_DIR.exists()}')

## 3. Загрузка кропов и базовый осмотр

In [ ]:
def load_crops(directory: Path) -> dict[str, np.ndarray]:
    """Загружает все изображения из папки. Ключ — имя файла."""
    crops = {}
    for p in sorted(directory.iterdir()):
        if p.suffix.lower() in IMG_EXTS:
            img = cv2.imread(str(p))
            if img is not None:
                crops[p.name] = img
    print(f'Загружено {len(crops)} кропов')
    return crops

crops = load_crops(CROPS_DIR)

In [ ]:
def show_crops_grid(crops: dict, max_show: int = 12, scale: int = 3):
    """Показывает кропы сеткой с размерами и LapVar."""
    names = list(crops.keys())[:max_show]
    n = len(names)
    cols = min(4, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * scale, rows * scale))
    axes = np.array(axes).flatten()
    for i, name in enumerate(names):
        img = crops[name]
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        lapvar = cv2.Laplacian(gray, cv2.CV_64F).var()
        axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[i].set_title(f'{name}\n{img.shape[1]}×{img.shape[0]}  LV={lapvar:.0f}', fontsize=7)
        axes[i].axis('off')
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    plt.suptitle('Все кропы (первые 12)', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()

show_crops_grid(crops)

## 4. Метрики качества изображения

Поскольку у нас нет ground truth (оригинальных чётких штрихкодов),
используем **reference-free** метрики:

| Метрика | Что измеряет | Лучше |
|---------|-------------|-------|
| **LapVar** | Резкость (дисперсия Лапласиана) | больше |
| **Tenengrad** | Резкость по Sobel | больше |
| **BRISQUE** | Натуральность изображения (НЕТ реализации без доп.пакета) | меньше |
| **Contrast** | RMS контраст | больше |
| **BarContrast** | Контраст между тёмными и светлыми полосами | больше |
| **EdgeDensity** | Плотность рёбер Canny | больше = более чёткие полосы |

In [ ]:
@dataclass
class ImageMetrics:
    lapvar: float = 0.0
    tenengrad: float = 0.0
    contrast: float = 0.0
    bar_contrast: float = 0.0
    edge_density: float = 0.0
    decoded_pyzbar: Optional[str] = None
    decoded_zxing: Optional[str] = None

    @property
    def decoded(self) -> bool:
        return self.decoded_pyzbar is not None or self.decoded_zxing is not None

    @property
    def decoded_value(self) -> Optional[str]:
        return self.decoded_pyzbar or self.decoded_zxing


def compute_metrics(img_bgr: np.ndarray) -> ImageMetrics:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float64)

    # Laplacian variance
    lapvar = cv2.Laplacian(gray, cv2.CV_64F).var()

    # Tenengrad (Sobel energy)
    sx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    tenengrad = float(np.mean(sx**2 + sy**2))

    # RMS contrast
    contrast = float(gray.std())

    # Bar contrast: разница между верхним и нижним квантилями (полосы vs фон)
    bar_contrast = float(np.percentile(gray, 90) - np.percentile(gray, 10))

    # Edge density
    edges = cv2.Canny(gray.astype(np.uint8), 50, 150)
    edge_density = float(edges.mean())

    # Декодирование
    img_uint8 = img_bgr if img_bgr.dtype == np.uint8 else (img_bgr * 255).astype(np.uint8)

    dec_pyzbar = None
    try:
        results = pyzbar.decode(img_uint8)
        if results:
            dec_pyzbar = results[0].data.decode('utf-8', errors='replace')
    except Exception:
        pass

    dec_zxing = None
    try:
        results = zxingcpp.read_barcodes(cv2.cvtColor(img_uint8, cv2.COLOR_BGR2RGB))
        if results:
            dec_zxing = results[0].text
    except Exception:
        pass

    return ImageMetrics(
        lapvar=lapvar, tenengrad=tenengrad, contrast=contrast,
        bar_contrast=bar_contrast, edge_density=edge_density,
        decoded_pyzbar=dec_pyzbar, decoded_zxing=dec_zxing
    )


print('✓ compute_metrics определён')

In [ ]:
# Базовые метрики на сырых кропах
baseline_metrics = {}
for name, img in tqdm(crops.items(), desc='baseline'):
    baseline_metrics[name] = compute_metrics(img)

df_baseline = pd.DataFrame([
    {
        'file': name,
        'w': crops[name].shape[1],
        'h': crops[name].shape[0],
        'lapvar': f'{m.lapvar:.1f}',
        'tenengrad': f'{m.tenengrad:.1f}',
        'contrast': f'{m.contrast:.1f}',
        'bar_contrast': f'{m.bar_contrast:.1f}',
        'edge_density': f'{m.edge_density:.2f}',
        'pyzbar': '✓' if m.decoded_pyzbar else '✗',
        'zxing': '✓' if m.decoded_zxing else '✗',
        'value': (m.decoded_value or '')[:20],
    }
    for name, m in baseline_metrics.items()
])

total = len(df_baseline)
pyzbar_ok = sum(1 for m in baseline_metrics.values() if m.decoded_pyzbar)
zxing_ok  = sum(1 for m in baseline_metrics.values() if m.decoded_zxing)
print(f'Baseline: pyzbar {pyzbar_ok}/{total}  |  zxing {zxing_ok}/{total}')
df_baseline

## 5. Выравнивание (Deskew)

Штрихкод в bbox от YOLO часто наклонён. Стратегия:
1. Найти доминирующий угол через **HoughLinesP** на краях Canny
2. Если Hough не уверен — fallback на **минимальный ограничивающий прямоугольник** тёмных пикселей
3. Повернуть изображение на найденный угол

In [ ]:
def deskew_hough(img_bgr: np.ndarray, debug: bool = False) -> tuple[np.ndarray, float]:
    """
    Выравнивает штрихкод через HoughLinesP.
    Возвращает (выровненное изображение, угол в градусах).
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Усиливаем края перед Hough
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(4, 4))
    enhanced = clahe.apply(gray)
    edges = cv2.Canny(enhanced, 30, 100, apertureSize=3)

    lines = cv2.HoughLinesP(
        edges, rho=1, theta=np.pi/360,
        threshold=20,
        minLineLength=img_bgr.shape[0] // 4,
        maxLineGap=8
    )

    angle = 0.0
    if lines is not None and len(lines) > 0:
        angles = []
        for line in lines:
            x1, y1, x2, y2 = line[0]
            a = np.degrees(np.arctan2(y2 - y1, x2 - x1))
            # Штрихкод вертикальные полосы → линии почти вертикальны (около ±90°)
            # или горизонтальные края — берём всё
            angles.append(a)

        # Берём угол с наибольшей плотностью в гистограмме
        hist, bins = np.histogram(angles, bins=72, range=(-90, 90))
        peak_bin = np.argmax(hist)
        angle = float((bins[peak_bin] + bins[peak_bin + 1]) / 2)

        # Нормализуем: если угол близок к ±90 — это вертикальные линии, 
        # поворачиваем на небольшой корректирующий угол
        if abs(angle) > 45:
            angle = angle - np.sign(angle) * 90

    if debug:
        print(f'  Hough угол: {angle:.2f}°  (из {len(lines) if lines is not None else 0} линий)')

    h, w = img_bgr.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    rotated = cv2.warpAffine(img_bgr, M, (w, h),
                              flags=cv2.INTER_CUBIC,
                              borderMode=cv2.BORDER_REPLICATE)
    return rotated, angle


def deskew_minrect(img_bgr: np.ndarray, debug: bool = False) -> tuple[np.ndarray, float]:
    """
    Fallback: выравнивание через минимальный ограничивающий прямоугольник
    тёмных пикселей (работает когда Hough не нашёл линий).
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = np.column_stack(np.where(binary > 0))
    if len(coords) < 10:
        return img_bgr, 0.0
    rect = cv2.minAreaRect(coords[:, ::-1].astype(np.float32))
    angle = rect[2]
    # minAreaRect возвращает угол от -90 до 0
    if angle < -45:
        angle += 90
    if debug:
        print(f'  minRect угол: {angle:.2f}°')
    h, w = img_bgr.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    rotated = cv2.warpAffine(img_bgr, M, (w, h),
                              flags=cv2.INTER_CUBIC,
                              borderMode=cv2.BORDER_REPLICATE)
    return rotated, angle


def deskew(img_bgr: np.ndarray, debug: bool = False) -> tuple[np.ndarray, float]:
    """Основной deskew: Hough с fallback на minRect."""
    result, angle = deskew_hough(img_bgr, debug=debug)
    if abs(angle) < 0.3:  # Hough не нашёл значимого угла — пробуем minRect
        result2, angle2 = deskew_minrect(img_bgr, debug=debug)
        if abs(angle2) > 0.3:
            return result2, angle2
    return result, angle


print('✓ deskew определён')

In [ ]:
# Визуализируем deskew на всех кропах
def show_deskew_results(crops: dict, max_show: int = 8):
    names = list(crops.keys())[:max_show]
    fig, axes = plt.subplots(len(names), 2, figsize=(8, len(names) * 2))
    if len(names) == 1:
        axes = [axes]
    for i, name in enumerate(names):
        img = crops[name]
        deskewed, angle = deskew(img)
        axes[i][0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[i][0].set_title(f'Original: {name}', fontsize=7)
        axes[i][0].axis('off')
        axes[i][1].imshow(cv2.cvtColor(deskewed, cv2.COLOR_BGR2RGB))
        axes[i][1].set_title(f'Deskewed ({angle:.1f}°)', fontsize=7)
        axes[i][1].axis('off')
    plt.suptitle('Deskew: до и после', fontsize=11)
    plt.tight_layout()
    plt.show()

show_deskew_results(crops)

## 6. Классические алгоритмы предобработки

Каждый алгоритм — это функция `img_bgr → img_bgr`.
Они будут компоноваться в пайплайны.

In [ ]:
# ── Алгоритмы предобработки ─────────────────────────────────────────────────

def preproc_identity(img: np.ndarray) -> np.ndarray:
    """Без обработки — baseline."""
    return img.copy()


def preproc_clahe(img: np.ndarray, clip: float = 3.0) -> np.ndarray:
    """CLAHE локальный контраст."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(4, 2))
    eq = clahe.apply(gray)
    return cv2.cvtColor(eq, cv2.COLOR_GRAY2BGR)


def preproc_unsharp(img: np.ndarray, sigma: float = 1.5, strength: float = 1.5) -> np.ndarray:
    """Unsharp mask — анизотропный (сильнее по X для вертикальных полос)."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (0, 0), sigmaX=sigma, sigmaY=sigma * 0.4)
    sharp = cv2.addWeighted(gray, 1.0 + strength, blurred, -strength, 0)
    return cv2.cvtColor(sharp, cv2.COLOR_GRAY2BGR)


def preproc_adaptive_thresh(img: np.ndarray, block: int = 15, C: int = 3) -> np.ndarray:
    """Адаптивный порог — бинаризация."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    binary = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, block, C
    )
    return cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)


def preproc_morph_close(img: np.ndarray, ksize: int = 3) -> np.ndarray:
    """Морфологическое закрытие — склеивает разорванные полосы."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, ksize))  # вертикальное ядро
    closed = cv2.morphologyEx(gray, cv2.MORPH_CLOSE, kernel)
    return cv2.cvtColor(closed, cv2.COLOR_GRAY2BGR)


def preproc_bilateral(img: np.ndarray) -> np.ndarray:
    """Bilateral filter — сглаживает шум сохраняя края."""
    result = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
    return result


def preproc_wiener_deconv(img: np.ndarray, psf_size: int = 5, noise_var: float = 0.01) -> np.ndarray:
    """
    Простая деконволюция Винера с Gaussian PSF.
    Полезна при defocus blur если знаем приблизительный радиус.
    """
    from scipy.signal import convolve2d
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float64) / 255.0

    # PSF — Gaussian
    psf = cv2.getGaussianKernel(psf_size, 0)
    psf = psf @ psf.T
    psf /= psf.sum()

    # Wiener в частотной области
    H = np.fft.fft2(psf, s=gray.shape)
    G = np.fft.fft2(gray)
    H_conj = np.conj(H)
    W = H_conj / (np.abs(H)**2 + noise_var)
    restored = np.real(np.fft.ifft2(W * G))
    restored = np.clip(restored, 0, 1)
    result = (restored * 255).astype(np.uint8)
    return cv2.cvtColor(result, cv2.COLOR_GRAY2BGR)


def preproc_1d_redraw(img: np.ndarray, top_frac: float = 0.6) -> np.ndarray:
    """
    1D-перерисовка: усредняем строки верхней части (без цифр),
    бинаризуем профиль и рисуем чистые вертикальные полосы.
    Работает только после deskew!
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    # Берём верхние top_frac строк
    top = gray[:int(h * top_frac), :]
    profile = top.mean(axis=0)  # 1D профиль

    # Адаптивная бинаризация 1D профиля
    p_norm = ((profile - profile.min()) / (profile.ptp() + 1e-6) * 255).astype(np.uint8)
    p_2d = p_norm.reshape(1, -1)
    _, binary_1d = cv2.threshold(p_2d, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    binary_col = binary_1d.flatten()

    # Рисуем чистый штрихкод
    canvas = np.ones((h, w), dtype=np.uint8) * 255
    canvas[int(h * 0.1):int(h * 0.85), :] = binary_col[np.newaxis, :]
    return cv2.cvtColor(canvas, cv2.COLOR_GRAY2BGR)


print('✓ все preproc-функции определены')

In [ ]:
# Визуализация каждой предобработки на одном примере
def show_preproc_comparison(img_bgr: np.ndarray, title: str = ''):
    steps = [
        ('Original',         preproc_identity(img_bgr)),
        ('CLAHE',            preproc_clahe(img_bgr)),
        ('Unsharp',          preproc_unsharp(img_bgr)),
        ('Adaptive Thresh',  preproc_adaptive_thresh(img_bgr)),
        ('Morph Close',      preproc_morph_close(img_bgr)),
        ('Bilateral',        preproc_bilateral(img_bgr)),
        ('Wiener Deconv',    preproc_wiener_deconv(img_bgr)),
        ('1D Redraw\n(after deskew)', preproc_1d_redraw(deskew(img_bgr)[0])),
    ]
    n = len(steps)
    fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(14, 5))
    axes = axes.flatten()
    for i, (name, result) in enumerate(steps):
        m = compute_metrics(result)
        decoded_str = f'✓ {(m.decoded_value or "")[:15]}' if m.decoded else '✗'
        axes[i].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
        axes[i].set_title(f'{name}\nLV={m.lapvar:.0f}  {decoded_str}', fontsize=7)
        axes[i].axis('off')
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    plt.suptitle(f'Предобработки: {title}', fontsize=10)
    plt.tight_layout()
    plt.show()


# Запускаем на первом кропе
first_name = list(crops.keys())[0]
show_preproc_comparison(crops[first_name], title=first_name)

In [ ]:
# Интерактивный выбор кропа для просмотра
# Измени INDEX чтобы посмотреть другой кроп
INDEX = 1
name = list(crops.keys())[INDEX]
print(f'Просматриваем: {name}')
show_preproc_comparison(crops[name], title=name)

## 7. SR-модели: EDSR, IFAN, DRBNet

Каждая модель оборачивается в тот же интерфейс `img_bgr → img_bgr`.
Модели загружаются лениво (при первом вызове).

In [ ]:
import torch
import torch.nn.functional as F_torch
from functools import lru_cache

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# ── EDSR (через super-image, не требует клонирования репо) ──────────────────
_edsr_model = None

def get_edsr():
    global _edsr_model
    if _edsr_model is None:
        from super_image import EdsrModel, ImageLoader
        _edsr_model = EdsrModel.from_pretrained('eugenesiow/edsr-base', scale=4)
        _edsr_model.eval()
        print('✓ EDSR загружен')
    return _edsr_model


def preproc_edsr(img_bgr: np.ndarray) -> np.ndarray:
    """EDSR x4 super-resolution."""
    from super_image import ImageLoader
    from skimage import img_as_ubyte
    from PIL import Image

    model = get_edsr()
    pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    inputs = ImageLoader.load_image(pil)
    with torch.no_grad():
        preds = model(inputs)
    sr_pil = ImageLoader.save_image(preds)
    return cv2.cvtColor(np.array(sr_pil), cv2.COLOR_RGB2BGR)


print('✓ EDSR wrapper определён')

In [ ]:
# ── IFAN (требует клонированного репо) ──────────────────────────────────────
# git clone https://github.com/codeslake/IFAN
# Скачать веса: https://drive.google.com/drive/folders/1Ld93_OIGXnCFlqjGRpCd13TSvmqH5MZ8
# Положить в IFAN/ckpt/IFAN_DPDD.pytorch

_ifan_model = None

def get_ifan():
    global _ifan_model
    if _ifan_model is None:
        if not IFAN_REPO.exists():
            raise FileNotFoundError(
                f'IFAN репо не найден: {IFAN_REPO}\n'
                'Выполни: git clone https://github.com/codeslake/IFAN'
            )
        import sys
        sys.path.insert(0, str(IFAN_REPO))
        from models.model import IFAN

        ckpt_path = IFAN_REPO / 'ckpt' / 'IFAN_DPDD.pytorch'
        if not ckpt_path.exists():
            raise FileNotFoundError(f'Веса не найдены: {ckpt_path}')

        model = IFAN()
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)
        model.to(DEVICE)
        model.eval()
        _ifan_model = model
        print('✓ IFAN загружен')
    return _ifan_model


def preproc_ifan(img_bgr: np.ndarray) -> np.ndarray:
    """
    IFAN defocus deblurring.
    """
    from skimage import img_as_ubyte
    model = get_ifan()

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    t = torch.from_numpy(img_rgb).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

    # Паддинг до кратного 16
    factor = 16
    _, _, h, w = t.shape
    ph = (factor - h % factor) % factor
    pw = (factor - w % factor) % factor
    t = F_torch.pad(t, (0, pw, 0, ph), 'reflect')

    with torch.no_grad():
        out = model(t)

    # Если модель возвращает tuple — берём первый элемент
    if isinstance(out, (list, tuple)):
        out = out[0]

    out = out[:, :, :h, :w]
    out = torch.clamp(out, 0, 1).cpu().squeeze(0).permute(1, 2, 0).numpy()
    return cv2.cvtColor(img_as_ubyte(out), cv2.COLOR_RGB2BGR)


print('✓ IFAN wrapper определён')

In [ ]:
# ── DRBNet (требует клонированного репо) ────────────────────────────────────
# git clone https://github.com/lingyanruan/DRBNet
# Веса: https://github.com/lingyanruan/DRBNet (ссылки в README)

_drbnet_model = None

def get_drbnet():
    global _drbnet_model
    if _drbnet_model is None:
        if not DRBNET_REPO.exists():
            raise FileNotFoundError(
                f'DRBNet репо не найден: {DRBNET_REPO}\n'
                'Выполни: git clone https://github.com/lingyanruan/DRBNet'
            )
        import sys
        sys.path.insert(0, str(DRBNET_REPO))
        from models.DRBNet import DRBNet

        ckpt_path = next(DRBNET_REPO.glob('**/*.pth'), None)
        if ckpt_path is None:
            raise FileNotFoundError(f'Веса .pth не найдены в {DRBNET_REPO}')
        print(f'  Загружаем веса: {ckpt_path.name}')

        model = DRBNet()
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        state = ckpt.get('model', ckpt.get('state_dict', ckpt))
        model.load_state_dict(state, strict=False)
        model.to(DEVICE)
        model.eval()
        _drbnet_model = model
        print('✓ DRBNet загружен')
    return _drbnet_model


def preproc_drbnet(img_bgr: np.ndarray) -> np.ndarray:
    """DRBNet defocus deblurring."""
    from skimage import img_as_ubyte
    model = get_drbnet()

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    t = torch.from_numpy(img_rgb).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

    factor = 8
    _, _, h, w = t.shape
    ph = (factor - h % factor) % factor
    pw = (factor - w % factor) % factor
    t = F_torch.pad(t, (0, pw, 0, ph), 'reflect')

    with torch.no_grad():
        out = model(t)
    if isinstance(out, (list, tuple)):
        out = out[-1]

    out = out[:, :, :h, :w]
    out = torch.clamp(out, 0, 1).cpu().squeeze(0).permute(1, 2, 0).numpy()
    return cv2.cvtColor(img_as_ubyte(out), cv2.COLOR_RGB2BGR)


print('✓ DRBNet wrapper определён')

## 8. Пайплайны — комбинации шагов

Каждый пайплайн = список функций `img → img`, применяемых последовательно.
Deskew всегда идёт первым если включён.

In [ ]:
from typing import NamedTuple

class Pipeline(NamedTuple):
    name: str
    steps: list  # list of Callable
    use_deskew: bool = True


def run_pipeline(img_bgr: np.ndarray, pipeline: Pipeline) -> np.ndarray:
    """Применяет пайплайн к изображению."""
    result = img_bgr.copy()
    if pipeline.use_deskew:
        result, _ = deskew(result)
    for step in pipeline.steps:
        result = step(result)
    return result


# ── Определяем все пайплайны ─────────────────────────────────────────────────
PIPELINES = [
    Pipeline('01_raw',
             [preproc_identity], use_deskew=False),

    Pipeline('02_deskew_only',
             [preproc_identity], use_deskew=True),

    Pipeline('03_clahe',
             [preproc_clahe], use_deskew=True),

    Pipeline('04_unsharp',
             [preproc_unsharp], use_deskew=True),

    Pipeline('05_bilateral+unsharp',
             [preproc_bilateral, preproc_unsharp], use_deskew=True),

    Pipeline('06_clahe+thresh',
             [preproc_clahe, preproc_adaptive_thresh], use_deskew=True),

    Pipeline('07_unsharp+thresh',
             [preproc_unsharp, preproc_adaptive_thresh], use_deskew=True),

    Pipeline('08_bilateral+clahe+thresh',
             [preproc_bilateral, preproc_clahe, preproc_adaptive_thresh], use_deskew=True),

    Pipeline('09_wiener',
             [preproc_wiener_deconv], use_deskew=True),

    Pipeline('10_wiener+thresh',
             [preproc_wiener_deconv, preproc_adaptive_thresh], use_deskew=True),

    Pipeline('11_morph+thresh',
             [preproc_morph_close, preproc_adaptive_thresh], use_deskew=True),

    Pipeline('12_1d_redraw',
             [preproc_1d_redraw], use_deskew=True),

    # ── Нейросетевые пайплайны (раскомментировать после загрузки моделей) ──
    # Pipeline('13_edsr',
    #          [preproc_edsr], use_deskew=True),
    # Pipeline('14_edsr+thresh',
    #          [preproc_edsr, preproc_adaptive_thresh], use_deskew=True),
    # Pipeline('15_ifan',
    #          [preproc_ifan], use_deskew=True),
    # Pipeline('16_ifan+thresh',
    #          [preproc_ifan, preproc_adaptive_thresh], use_deskew=True),
    # Pipeline('17_drbnet',
    #          [preproc_drbnet], use_deskew=True),
    # Pipeline('18_drbnet+thresh',
    #          [preproc_drbnet, preproc_adaptive_thresh], use_deskew=True),
]

print(f'Определено {len(PIPELINES)} пайплайнов')
for p in PIPELINES:
    steps_str = ' → '.join(f.__name__.replace("preproc_", "") for f in p.steps)
    deskew_str = '[deskew] → ' if p.use_deskew else ''
    print(f'  {p.name}: {deskew_str}{steps_str}')

## 9. Массовый прогон всех пайплайнов

In [ ]:
@dataclass
class PipelineResult:
    crop_name: str
    pipeline_name: str
    metrics: ImageMetrics
    result_img: np.ndarray
    error: Optional[str] = None


def evaluate_all(
    crops: dict,
    pipelines: list,
    skip_nn: bool = True   # пропускать нейросетевые если не загружены
) -> list[PipelineResult]:
    results = []
    for name, img in tqdm(crops.items(), desc='crops'):
        for pipeline in pipelines:
            try:
                result_img = run_pipeline(img, pipeline)
                m = compute_metrics(result_img)
                results.append(PipelineResult(
                    crop_name=name,
                    pipeline_name=pipeline.name,
                    metrics=m,
                    result_img=result_img
                ))
            except Exception as e:
                results.append(PipelineResult(
                    crop_name=name,
                    pipeline_name=pipeline.name,
                    metrics=ImageMetrics(),
                    result_img=img,
                    error=str(e)[:80]
                ))
    return results


print('✓ evaluate_all определён')

In [ ]:
# !! ЗАПУСКАЕТ ВСЕ ПАЙПЛАЙНЫ — может занять несколько минут !!
all_results = evaluate_all(crops, PIPELINES)
print(f'Всего результатов: {len(all_results)}')

## 10. Итоговая таблица результатов

In [ ]:
def results_to_df(results: list[PipelineResult]) -> pd.DataFrame:
    rows = []
    for r in results:
        m = r.metrics
        rows.append({
            'crop': r.crop_name,
            'pipeline': r.pipeline_name,
            'pyzbar': '✓' if m.decoded_pyzbar else '✗',
            'zxing':  '✓' if m.decoded_zxing  else '✗',
            'decoded': m.decoded,
            'value': (m.decoded_value or '')[:25],
            'lapvar': round(m.lapvar, 1),
            'tenengrad': round(m.tenengrad, 1),
            'bar_contrast': round(m.bar_contrast, 1),
            'edge_density': round(m.edge_density, 2),
            'error': r.error or '',
        })
    return pd.DataFrame(rows)


df = results_to_df(all_results)

# Сводная таблица: для каждого пайплайна — сколько раскодировано
summary = (
    df.groupby('pipeline')
    .agg(
        decoded_count=('decoded', 'sum'),
        total=('decoded', 'count'),
        pyzbar_ok=('pyzbar', lambda x: (x == '✓').sum()),
        zxing_ok=('zxing',  lambda x: (x == '✓').sum()),
        avg_lapvar=('lapvar', 'mean'),
        avg_bar_contrast=('bar_contrast', 'mean'),
        avg_edge_density=('edge_density', 'mean'),
    )
    .assign(decode_rate=lambda x: (x.decoded_count / x.total * 100).round(1))
    .sort_values('decoded_count', ascending=False)
    .reset_index()
)

print('=== СВОДКА ПО ПАЙПЛАЙНАМ ===')
summary

In [ ]:
# Сохраняем полную таблицу
df.to_csv(RESULTS_DIR / 'full_results.csv', index=False)
summary.to_csv(RESULTS_DIR / 'summary.csv', index=False)
print(f'Сохранено в {RESULTS_DIR}')

# Только успешно декодированные
decoded_df = df[df['decoded']].sort_values('pipeline')
print(f'\nВсего успешных декодирований: {len(decoded_df)}')
decoded_df[['crop', 'pipeline', 'pyzbar', 'zxing', 'value', 'lapvar']]

## 11. Визуализация результатов

In [ ]:
def plot_summary_bar(summary: pd.DataFrame):
    """График decode_rate по пайплайнам."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Decode rate
    colors = ['#2ecc71' if r > 0 else '#e74c3c'
              for r in summary['decode_rate']]
    axes[0].barh(summary['pipeline'], summary['decode_rate'], color=colors)
    axes[0].set_xlabel('Decode rate (%)')
    axes[0].set_title('Процент успешного декодирования по пайплайнам')
    axes[0].axvline(x=summary['decode_rate'].iloc[-1], color='gray',
                    linestyle='--', alpha=0.5, label='baseline (raw)')
    axes[0].legend(fontsize=8)

    # Avg LapVar
    axes[1].barh(summary['pipeline'], summary['avg_lapvar'], color='#3498db', alpha=0.8)
    axes[1].set_xlabel('Avg Laplacian Variance')
    axes[1].set_title('Средняя резкость по пайплайнам')

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'summary_chart.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_summary_bar(summary)

In [ ]:
def show_pipeline_strip(crop_name: str, results: list[PipelineResult],
                         max_pipelines: int = 8):
    """
    Показывает все пайплайны для одного кропа в одну строку.
    Зелёная рамка = декодировано.
    """
    crop_results = [r for r in results if r.crop_name == crop_name][:max_pipelines]
    n = len(crop_results)
    fig, axes = plt.subplots(1, n, figsize=(n * 2.5, 3))
    if n == 1:
        axes = [axes]

    for i, r in enumerate(crop_results):
        m = r.metrics
        axes[i].imshow(cv2.cvtColor(r.result_img, cv2.COLOR_BGR2RGB))
        decoded_str = '✓' if m.decoded else '✗'
        color = '#2ecc71' if m.decoded else '#e74c3c'
        axes[i].set_title(
            f'{r.pipeline_name}\n{decoded_str} LV={m.lapvar:.0f}',
            fontsize=6, color=color
        )
        for spine in axes[i].spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(2.5)
        axes[i].axis('off')

    plt.suptitle(f'Кроп: {crop_name}', fontsize=9)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'strip_{crop_name}.png', dpi=150, bbox_inches='tight')
    plt.show()


# Показываем для первых 3 кропов
for name in list(crops.keys())[:3]:
    show_pipeline_strip(name, all_results)

In [ ]:
def show_best_worst(results: list[PipelineResult], n: int = 3):
    """
    Лучшие и худшие случаи: сравниваем raw vs лучший пайплайн для каждого кропа.
    """
    # Для каждого кропа найдём лучший пайплайн (по decode + lapvar)
    per_crop = {}
    for r in results:
        if r.crop_name not in per_crop:
            per_crop[r.crop_name] = {'raw': None, 'best': None, 'best_score': -1}
        if r.pipeline_name == '01_raw':
            per_crop[r.crop_name]['raw'] = r
        score = (10000 if r.metrics.decoded else 0) + r.metrics.lapvar
        if score > per_crop[r.crop_name]['best_score']:
            per_crop[r.crop_name]['best_score'] = score
            per_crop[r.crop_name]['best'] = r

    # Сортируем по улучшению
    improvements = sorted(
        per_crop.items(),
        key=lambda x: x[1]['best_score'],
        reverse=True
    )[:n]

    fig, axes = plt.subplots(n, 2, figsize=(8, n * 2.5))
    if n == 1:
        axes = [axes]

    for i, (crop_name, data) in enumerate(improvements):
        raw = data['raw']
        best = data['best']
        for j, (r, label) in enumerate([(raw, 'RAW'), (best, 'BEST')]):
            if r is None:
                continue
            m = r.metrics
            color = '#2ecc71' if m.decoded else '#e74c3c'
            decoded_str = f'✓ {(m.decoded_value or "")[:12]}' if m.decoded else '✗'
            axes[i][j].imshow(cv2.cvtColor(r.result_img, cv2.COLOR_BGR2RGB))
            axes[i][j].set_title(
                f'[{label}] {r.pipeline_name}\n{decoded_str}  LV={m.lapvar:.0f}',
                fontsize=7, color=color
            )
            axes[i][j].axis('off')

    plt.suptitle(f'Топ-{n}: raw vs лучший пайплайн', fontsize=10)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'best_vs_raw.png', dpi=150, bbox_inches='tight')
    plt.show()


show_best_worst(all_results, n=min(5, len(crops)))

## 12. Анализ deskew — распределение углов

In [ ]:
angles_data = {}
for name, img in crops.items():
    _, angle_h = deskew_hough(img)
    _, angle_m = deskew_minrect(img)
    angles_data[name] = {'hough': angle_h, 'minrect': angle_m}

angles_df = pd.DataFrame(angles_data).T

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(angles_df['hough'], bins=20, color='#3498db', alpha=0.7, label='Hough')
axes[0].hist(angles_df['minrect'], bins=20, color='#e67e22', alpha=0.7, label='MinRect')
axes[0].set_xlabel('Угол (°)')
axes[0].set_ylabel('Кол-во кропов')
axes[0].set_title('Распределение обнаруженных углов наклона')
axes[0].legend()

axes[1].scatter(angles_df['hough'], angles_df['minrect'], alpha=0.6)
axes[1].plot([-45, 45], [-45, 45], 'r--', alpha=0.5)
axes[1].set_xlabel('Hough угол')
axes[1].set_ylabel('MinRect угол')
axes[1].set_title('Согласованность методов определения угла')

plt.tight_layout()
plt.show()
print(angles_df.describe())

## 13. Deep dive — детальный анализ одного кропа

Пошаговая визуализация с профилями яркости.

In [ ]:
def deep_dive(img_bgr: np.ndarray, name: str = ''):
    """
    Детальный анализ одного кропа:
    - исходное + deskewed
    - горизонтальный профиль яркости (до/после)
    - лучший классический пайплайн
    - гистограмма
    """
    deskewed, angle = deskew(img_bgr)
    gray_orig = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray_desk = cv2.cvtColor(deskewed, cv2.COLOR_BGR2GRAY)

    fig = plt.figure(figsize=(14, 8))
    gs = gridspec.GridSpec(2, 4, figure=fig)

    # Исходное
    ax0 = fig.add_subplot(gs[0, 0])
    ax0.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    m0 = compute_metrics(img_bgr)
    ax0.set_title(f'Original\nLV={m0.lapvar:.0f}  {"✓" if m0.decoded else "✗"}', fontsize=8)
    ax0.axis('off')

    # Deskewed
    ax1 = fig.add_subplot(gs[0, 1])
    ax1.imshow(cv2.cvtColor(deskewed, cv2.COLOR_BGR2RGB))
    m1 = compute_metrics(deskewed)
    ax1.set_title(f'Deskewed ({angle:.1f}°)\nLV={m1.lapvar:.0f}  {"✓" if m1.decoded else "✗"}', fontsize=8)
    ax1.axis('off')

    # Лучший классический
    best_classic = preproc_bilateral(preproc_clahe(deskewed))
    best_thresh = preproc_adaptive_thresh(best_classic)
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.imshow(cv2.cvtColor(best_classic, cv2.COLOR_BGR2RGB))
    m2 = compute_metrics(best_classic)
    ax2.set_title(f'Bilateral+CLAHE\nLV={m2.lapvar:.0f}  {"✓" if m2.decoded else "✗"}', fontsize=8)
    ax2.axis('off')

    ax3 = fig.add_subplot(gs[0, 3])
    ax3.imshow(cv2.cvtColor(best_thresh, cv2.COLOR_BGR2RGB))
    m3 = compute_metrics(best_thresh)
    ax3.set_title(f'+AdaptiveThresh\nLV={m3.lapvar:.0f}  {"✓" if m3.decoded else "✗"}', fontsize=8)
    ax3.axis('off')

    # 1D профиль — до и после deskew
    ax4 = fig.add_subplot(gs[1, :2])
    h = gray_orig.shape[0]
    profile_orig = gray_orig[int(h*0.1):int(h*0.7), :].mean(axis=0)
    profile_desk = gray_desk[int(h*0.1):int(h*0.7), :].mean(axis=0)
    ax4.plot(profile_orig, alpha=0.6, label='Original', color='#e74c3c')
    ax4.plot(profile_desk, alpha=0.8, label='Deskewed', color='#2ecc71')
    ax4.set_title('1D яркостный профиль (среднее по строкам)', fontsize=8)
    ax4.set_xlabel('Пиксель X')
    ax4.set_ylabel('Яркость')
    ax4.legend(fontsize=7)
    ax4.grid(alpha=0.3)

    # Гистограмма
    ax5 = fig.add_subplot(gs[1, 2:])
    for gray, label, color in [
        (gray_orig, 'Original', '#e74c3c'),
        (gray_desk, 'Deskewed', '#3498db'),
        (cv2.cvtColor(best_thresh, cv2.COLOR_BGR2GRAY), '+Thresh', '#2ecc71'),
    ]:
        ax5.hist(gray.flatten(), bins=64, alpha=0.5, label=label, color=color, density=True)
    ax5.set_title('Гистограммы яркости', fontsize=8)
    ax5.legend(fontsize=7)
    ax5.grid(alpha=0.3)

    plt.suptitle(f'Deep Dive: {name}', fontsize=10)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'deepdive_{name}.png', dpi=150, bbox_inches='tight')
    plt.show()


# Запускаем на первых 2 кропах
for name in list(crops.keys())[:2]:
    deep_dive(crops[name], name=name)

## 14. Итоговый отчёт

In [ ]:
total_crops = len(crops)
total_pipelines = len(PIPELINES)

best_pipeline = summary.iloc[0]
baseline_row = summary[summary['pipeline'] == '01_raw'].iloc[0]

print('=' * 60)
print('ИТОГОВЫЙ ОТЧЁТ')
print('=' * 60)
print(f'Кропов: {total_crops}')
print(f'Пайплайнов: {total_pipelines}')
print()
print(f'BASELINE (raw):')
print(f'  pyzbar: {int(baseline_row.pyzbar_ok)}/{total_crops}')
print(f'  zxing:  {int(baseline_row.zxing_ok)}/{total_crops}')
print(f'  decode rate: {baseline_row.decode_rate}%')
print()
print(f'ЛУЧШИЙ ПАЙПЛАЙН: {best_pipeline.pipeline}')
print(f'  pyzbar: {int(best_pipeline.pyzbar_ok)}/{total_crops}')
print(f'  zxing:  {int(best_pipeline.zxing_ok)}/{total_crops}')
print(f'  decode rate: {best_pipeline.decode_rate}%')
print(f'  avg LapVar: {best_pipeline.avg_lapvar:.1f}')
print()
improvement = best_pipeline.decode_rate - baseline_row.decode_rate
print(f'Улучшение: +{improvement:.1f}%')
print('=' * 60)